# SU(3) historical coefficient recovery R2
Strict title/author validation, unique hashes, and targeted OCR of the KPS definition/table pages.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','pymupdf','pytesseract','pillow'])
print('dependencies ready')

In [ ]:
SOURCE='#!/usr/bin/env python3\n"""\nStrict R2 recovery for SU(3) O(y^5)/O(y^6) historical coefficients.\n\nThis revision fixes the false-positive behavior of the first extractor:\n  * title/author validation is mandatory;\n  * one PDF hash cannot fill multiple source slots;\n  * hep-lat/0005009 is explicitly recognized as the 2000 GFMC paper;\n  * the KPS scan receives targeted full-page OCR for its definition and table pages.\n\nNo coefficient is accepted automatically.\n"""\nfrom __future__ import annotations\n\nimport hashlib, json, os, re, subprocess, sys\nfrom pathlib import Path\n\ntry:\n    import fitz\nexcept Exception:\n    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])\n    import fitz\n\nBASE = Path("/content") if Path("/content").exists() else Path("/mnt/data")\nOUT = BASE / "SU3_Y5_Y6_HISTORICAL_RECOVERY_R2"\nOUT.mkdir(parents=True, exist_ok=True)\n\nSOURCES = {\n    "KPS_1981_string": {\n        "required": [["kogut"], ["pearson"], ["shigemitsu"]],\n        "title_terms": ["string tension", "roughening", "su(3)"],\n        "expected_pages_min": 6,\n    },\n    "HIP_1986_string": {\n        "required": [["hamer"], ["irving"], ["preece"]],\n        "title_terms": ["cluster expansion", "su(3)"],\n        "expected_pages_min": 5,\n    },\n    "HAMER_1989_mass": {\n        "required": [["hamer"]],\n        "title_terms": ["strong coupling", "glueball masses", "su(3)"],\n        "expected_pages_min": 4,\n    },\n}\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,"rb") as f:\n        for b in iter(lambda:f.read(1<<20),b""):\n            h.update(b)\n    return h.hexdigest()\n\ndef roots():\n    out=[BASE,Path("/mnt/data")]\n    if Path("/content/drive").exists():\n        out.append(Path("/content/drive"))\n    return out\n\ndef pdfs():\n    seen={}\n    for root in roots():\n        if not root.exists(): continue\n        for p in root.rglob("*.pdf"):\n            try: seen[sha256(p)] = p\n            except Exception: pass\n    return list(seen.values())\n\ndef extract_text(path):\n    doc=fitz.open(path)\n    pages=[p.get_text("text") for p in doc]\n    return pages\n\ndef normalized(text):\n    return re.sub(r"\\s+"," ",text.lower())\n\ndef validate(path, spec):\n    pages=extract_text(path)\n    head=normalized(" ".join(pages[:3]))\n    if "green’s function monte carlo" in head or "green\'s function monte carlo" in head:\n        return False, "identified as 2000 GFMC paper", pages\n    for group in spec["required"]:\n        if not any(term in head for term in group):\n            return False, f"missing required author group {group}", pages\n    hits=sum(term in head for term in spec["title_terms"])\n    if hits < max(1, len(spec["title_terms"])-1):\n        return False, f"title-term score too low ({hits})", pages\n    if len(pages) < spec["expected_pages_min"]:\n        return False, f"too few pages ({len(pages)})", pages\n    return True, "validated", pages\n\ndef score(path, spec):\n    name=path.name.lower()\n    pages=extract_text(path)\n    head=normalized(" ".join(pages[:3]))\n    s=0\n    for group in spec["required"]:\n        if any(t in head for t in group): s += 10\n    for t in spec["title_terms"]:\n        if t in head: s += 6\n    if "green\'s function monte carlo" in head or "green’s function monte carlo" in head:\n        s -= 100\n    return s\n\ndef upload():\n    if not Path("/content").exists(): return\n    try:\n        from google.colab import files\n    except Exception:\n        return\n    print("Upload missing historical PDFs. The KPS scan may be named LIT_Y5_8010101.pdf.")\n    for n,d in files.upload().items():\n        Path("/content",Path(n).name).write_bytes(d)\n\ndef ocr_page(page, scale=4.0):\n    try:\n        import pytesseract\n        from PIL import Image\n    except Exception:\n        subprocess.check_call([sys.executable,"-m","pip","install","-q","pytesseract","pillow"])\n        import pytesseract\n        from PIL import Image\n    if subprocess.call(["bash","-lc","command -v tesseract >/dev/null 2>&1"]) != 0:\n        subprocess.check_call(["bash","-lc","apt-get update -qq && apt-get install -y -qq tesseract-ocr"])\n    pix=page.get_pixmap(matrix=fitz.Matrix(scale,scale),alpha=False)\n    img=Image.frombytes("RGB",[pix.width,pix.height],pix.samples)\n    return pytesseract.image_to_string(img)\n\ndef targeted_kps(path):\n    doc=fitz.open(path)\n    outputs=[]\n    # Definitions are early; exact tables and figure captions tend to be late in the scan.\n    candidates=sorted(set([0,1,2,3,4,5] + list(range(max(0,len(doc)-6),len(doc)))))\n    for i in candidates:\n        embedded=doc[i].get_text("text")\n        ocr=ocr_page(doc[i],4.0)\n        outputs.append({\n            "page":i+1,\n            "embedded_text":embedded,\n            "ocr_text":ocr,\n            "signals":{\n                "table_I":bool(re.search(r"table\\s+i\\b",ocr,re.I)),\n                "table_II":bool(re.search(r"table\\s+ii\\b",ocr,re.I)),\n                "defines_x":bool(re.search(r"\\bx\\s*=",ocr,re.I)),\n                "defines_T":bool(re.search(r"\\bT\\s*=",ocr)),\n                "contains_x5_x6":("x5" in ocr.replace("^","").lower() or "x6" in ocr.replace("^","").lower()),\n            }\n        })\n    return outputs\n\ndef main():\n    allpdf=pdfs()\n    chosen={}\n    used=set()\n    diagnostics={}\n    for key,spec in SOURCES.items():\n        ranked=sorted([(score(p,spec),p) for p in allpdf], key=lambda x:(-x[0],str(x[1])))\n        accepted=None\n        rejects=[]\n        for sc,p in ranked:\n            h=sha256(p)\n            if h in used:\n                rejects.append((str(p),sc,"hash already used by another slot"))\n                continue\n            ok,reason,pages=validate(p,spec)\n            if ok:\n                accepted=(p,pages)\n                used.add(h)\n                break\n            rejects.append((str(p),sc,reason))\n        diagnostics[key]={"rejects":rejects[:20]}\n        if accepted:\n            chosen[key]=accepted[0]\n    missing=[k for k in SOURCES if k not in chosen]\n    if missing:\n        upload()\n        # One retry after upload.\n        allpdf=pdfs()\n        chosen={}\n        used=set()\n        diagnostics={}\n        for key,spec in SOURCES.items():\n            ranked=sorted([(score(p,spec),p) for p in allpdf], key=lambda x:(-x[0],str(x[1])))\n            rejects=[]\n            for sc,p in ranked:\n                h=sha256(p)\n                if h in used:\n                    rejects.append((str(p),sc,"hash already used by another slot"))\n                    continue\n                ok,reason,pages=validate(p,spec)\n                if ok:\n                    chosen[key]=p\n                    used.add(h)\n                    break\n                rejects.append((str(p),sc,reason))\n            diagnostics[key]={"rejects":rejects[:20]}\n    missing=[k for k in SOURCES if k not in chosen]\n\n    result={\n        "status":"PARTIAL" if missing else "SOURCES_VALIDATED",\n        "chosen":{k:{"path":str(p),"sha256":sha256(p),"pages":len(fitz.open(p))}\n                  for k,p in chosen.items()},\n        "missing":missing,\n        "diagnostics":diagnostics,\n        "accepted_coefficients":{},\n    }\n\n    if "KPS_1981_string" in chosen:\n        result["KPS_targeted_pages"]=targeted_kps(chosen["KPS_1981_string"])\n\n    j=OUT/"SU3_Y5_Y6_HISTORICAL_RECOVERY_R2.json"\n    j.write_text(json.dumps(result,indent=2),encoding="utf-8")\n\n    lines=[\n        "# SU(3) historical coefficient recovery R2",\n        "",\n        f"**Status:** `{result[\'status\']}`",\n        "",\n        "## Validated sources",\n        "",\n    ]\n    for k,v in result["chosen"].items():\n        lines += [f"- `{k}`: `{v[\'path\']}` — SHA-256 `{v[\'sha256\']}` — {v[\'pages\']} pages"]\n    lines += ["","## Missing sources",""]\n    lines += [f"- `{k}`" for k in missing] or ["- none"]\n    lines += ["","## Acceptance status","",\n              "No fifth- or sixth-order coefficient has been accepted automatically.",\n              "The KPS targeted OCR is stored in the JSON ledger for exact table reconstruction."]\n    m=OUT/"SU3_Y5_Y6_HISTORICAL_RECOVERY_R2.md"\n    m.write_text("\\n".join(lines),encoding="utf-8")\n    print("STATUS",result["status"])\n    print("chosen",result["chosen"])\n    print("missing",missing)\n    print("JSON",j)\n    print("MD",m)\n\nif __name__=="__main__":\n    main()\n'
from pathlib import Path
p=Path('/content/ENGINE_Y6_su3_y5_historical_recovery_r2.py')
p.write_text(SOURCE,encoding='utf-8')
exec(compile(SOURCE,str(p),'exec'),{'__name__':'__main__'})

In [ ]:
from pathlib import Path
import zipfile
out=Path('/content/SU3_Y5_Y6_HISTORICAL_RECOVERY_R2')
zp=Path('/content/SU3_Y5_Y6_HISTORICAL_RECOVERY_R2_RESULTS.zip')
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(out.parent))
print(zp)